In [1]:
import sys
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras as keras
sys.modules["tensorflow.keras"] = keras
sys.modules["keras"] = keras

import keras_tuner as kt
import qkeras
from qkeras import QDense, QConv2D, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

import tensorflow as tf
#from tensorflow import keras
#from tensorflow.keras import layers
#from keras.models import Sequential
#from keras.layers import MaxPooling2D, Activation, Flatten, Dropout
from tf_keras.models import Sequential
from tf_keras.layers import MaxPooling2D, Activation, Flatten, Dropout
import numpy as np

2026-04-21 00:11:36.208515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776723096.224531   12499 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776723096.229331   12499 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-21 00:11:36.248390: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from tensorflow.python.client import device_lib
print([x.name for x in device_lib.list_local_devices() if 'GPU' in x.name])

['/device:GPU:0']


I0000 00:00:1776723102.852336   12499 gpu_device.cc:2022] Created device /device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [3]:
import tensorflow as tf

# This makes TF tell us exactly what it's doing
tf.debugging.set_log_device_placement(True)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ RTX 4060 CONNECTED!")
    # Important: Prevents WSL2 from 'freezing' the GPU memory
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("❌ STILL FAIL: Checking for missing library names...")
    # This command below will show the actual missing file name
    import os
    os.system("ldd " + tf.__file__.replace("__init__.py", "python/_pywrap_tf_session.so") + " | grep 'not found'")

✅ RTX 4060 CONNECTED!


In [3]:
input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

X_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


In [4]:
#import tf_keras as keras

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=2,
    restore_best_weights=True
)

In [ ]:
#import tf_keras as keras
#from tensorflow import keras
#import tensorflow as tf
#import keras_tuner as kt
#from qkeras import QDense, QConv2D, QActivation, quantized_bits, quantized_relu

def build_model(hp):
    hp_bw = hp.Choice('bits', [4, 6, 8])
    hp_filters_1 = hp.Int('filters_1', min_value=16, max_value=64, step=16)
    hp_filters_2 = hp.Int('filters_2', min_value=16, max_value=64, step=16)
    hp_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    input_layer = keras.layers.Input(shape=(28, 28, 1))

    x = QConv2D(
    filters=hp_filters_1,
    kernel_size=(3, 3),
    name='qconv1',
    kernel_quantizer=quantized_bits(hp_bw, 0, alpha=1),
    bias_quantizer=quantized_bits(hp_bw, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
    )(input_layer)
    
    x = QActivation(activation=quantized_relu(hp_bw), name='relu1')(x)
    x = keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool1")(x)
    
    x = QConv2D(
    filters=hp_filters_2,
    kernel_size=(3, 3),
    name='qconv2',
    kernel_quantizer=quantized_bits(hp_bw, 0, alpha=1),
    bias_quantizer=quantized_bits(hp_bw, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
    )(x)
    
    x = QActivation(activation=quantized_relu(hp_bw), name='relu2')(x)
    x = keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool2")(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dropout(0.4)(x)
    
    x = QDense(
    10,
    name='qdense1',
    kernel_quantizer=quantized_bits(hp_bw, 0, alpha=1),
    bias_quantizer=quantized_bits(hp_bw, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
    )(x)
    
    output_layer = keras.layers.Activation(activation='softmax', name='softmax')(x)
    
    model = keras.Model(inputs=input_layer, outputs=output_layer)
    model.compile(loss="categorical_crossentropy", optimizer=keras.optimizers.Adam(learning_rate=hp_lr), metrics=["accuracy"])

    return model

tuner = kt.BayesianOptimization(build_model, objective='val_accuracy', project_name='mnist_qkeras_tuning', max_trials=15, executions_per_trial=1, overwrite=True)

In [20]:
tuner.search(X_train, y_train, batch_size=512, epochs=50, validation_split=0.1,callbacks=[early_stop],shuffle=True)

Trial 15 Complete [00h 03m 05s]
val_accuracy: 0.987666666507721

Best val_accuracy So Far: 0.9900000095367432
Total elapsed time: 00h 20m 17s


In [21]:
tuner.results_summary(num_trials=10)

Results summary
Results in ./mnist_qkeras_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 06 summary
Hyperparameters:
bits: 8
filters_1: 32
filters_2: 32
learning_rate: 0.001
Score: 0.9900000095367432

Trial 07 summary
Hyperparameters:
bits: 6
filters_1: 64
filters_2: 32
learning_rate: 0.01
Score: 0.9900000095367432

Trial 03 summary
Hyperparameters:
bits: 4
filters_1: 64
filters_2: 48
learning_rate: 0.01
Score: 0.9896666407585144

Trial 11 summary
Hyperparameters:
bits: 6
filters_1: 48
filters_2: 48
learning_rate: 0.001
Score: 0.9896666407585144

Trial 02 summary
Hyperparameters:
bits: 8
filters_1: 32
filters_2: 64
learning_rate: 0.001
Score: 0.9894999861717224

Trial 12 summary
Hyperparameters:
bits: 8
filters_1: 64
filters_2: 64
learning_rate: 0.01
Score: 0.9891666769981384

Trial 09 summary
Hyperparameters:
bits: 4
filters_1: 32
filters_2: 48
learning_rate: 0.01
Score: 0.9888333082199097

Trial 04 summary
Hyperparameters:
bits: 8
filters_1: 16
f

Num GPUs Available:  0


In [6]:
#import tf_keras as keras
#import tensorflow as tf
#from qkeras import QDense, QConv2D, QActivation, quantized_bits, quantized_relu


input_layer = keras.layers.Input(shape=(28, 28, 1))

x = QConv2D(
    filters=32,
    kernel_size=(3, 3),
    name='qconv1',
    kernel_quantizer=quantized_bits(6, 0, alpha=1),
    bias_quantizer=quantized_bits(6, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(input_layer)

x = QActivation(activation=quantized_relu(6), name='relu1')(x)
x = keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool1")(x)
x = QConv2D(
    filters=32,
    kernel_size=(3, 3),
    name='qconv2',
    kernel_quantizer=quantized_bits(6, 0, alpha=1),
    bias_quantizer=quantized_bits(6, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(x)

x = QActivation(activation=quantized_relu(6), name='relu2')(x)
x = keras.layers.MaxPooling2D(pool_size=(2, 2), name="pool2")(x)

x = keras.layers.Flatten()(x)
x = keras.layers.Dropout(0.4)(x)

x = QDense(
    10,
    name='qdense1',
    kernel_quantizer=quantized_bits(6, 0, alpha=1),
    bias_quantizer=quantized_bits(6, 0, alpha=1),
    kernel_initializer='lecun_uniform',
    kernel_regularizer=keras.regularizers.l1(1e-4)
)(x)

output_layer = keras.layers.Activation(activation='softmax', name='softmax')(x)
model = keras.Model(inputs=input_layer, outputs=output_layer)

model.summary()

I0000 00:00:1776721650.497661    7318 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 qconv1 (QConv2D)            (None, 26, 26, 32)        320       
                                                                 
 relu1 (QActivation)         (None, 26, 26, 32)        0         
                                                                 
 pool1 (MaxPooling2D)        (None, 13, 13, 32)        0         
                                                                 
 qconv2 (QConv2D)            (None, 11, 11, 32)        9248      
                                                                 
 relu2 (QActivation)         (None, 11, 11, 32)        0         
                                                                 
 pool2 (MaxPooling2D)        (None, 5, 5, 32)          0     

In [7]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [8]:
model.fit(X_train, y_train, batch_size=128, epochs=15, validation_split=0.1,shuffle=True)

Epoch 1/15


E0000 00:00:1776721662.005428    7318 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape ingradient_tape/model/relu2/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1776721662.676464    7560 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1776721663.809272    7563 service.cc:148] XLA service 0x75fc012d5600 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776721663.809658    7563 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-04-20 23:47:43.826987: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776721663.947690    7563 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


422/422 [==============================] - 8s 12ms/step - loss: 0.5079 - accuracy: 0.8817 - val_loss: 0.1992 - val_accuracy: 0.9750
Epoch 2/15
422/422 [==============================] - 4s 11ms/step - loss: 0.2283 - accuracy: 0.9619 - val_loss: 0.1658 - val_accuracy: 0.9823
Epoch 3/15
422/422 [==============================] - 4s 10ms/step - loss: 0.1968 - accuracy: 0.9699 - val_loss: 0.1516 - val_accuracy: 0.9833
Epoch 4/15
422/422 [==============================] - 4s 10ms/step - loss: 0.1811 - accuracy: 0.9734 - val_loss: 0.1408 - val_accuracy: 0.9873
Epoch 5/15
422/422 [==============================] - 5s 11ms/step - loss: 0.1700 - accuracy: 0.9762 - val_loss: 0.1383 - val_accuracy: 0.9858
Epoch 6/15
422/422 [==============================] - 5s 11ms/step - loss: 0.1633 - accuracy: 0.9774 - val_loss: 0.1332 - val_accuracy: 0.9875
Epoch 7/15
422/422 [==============================] - 5s 11ms/step - loss: 0.1591 - accuracy: 0.9781 - val_loss: 0.1305 - val_accuracy: 0.9877
Epoch 8/15

In [35]:
from qkeras.utils import load_qmodel


model = load_qmodel('/home/slopin/DAT255-project/Modeller/mnist_1_Qkeras_6bw.h5')

model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 qconv1 (QConv2D)            (None, 26, 26, 32)        320       
                                                                 
 relu1 (QActivation)         (None, 26, 26, 32)        0         
                                                                 
 pool1 (MaxPooling2D)        (None, 13, 13, 32)        0         
                                                                 
 qconv2 (QConv2D)            (None, 11, 11, 32)        9248      
                                                                 
 relu2 (QActivation)         (None, 11, 11, 32)        0         
                                                                 
 pool2 (MaxPooling2D)        (None, 5, 5, 32)          0   

In [33]:
score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.12265104055404663
Test accuracy: 0.9883000254631042


In [34]:
model.save('/home/slopin/DAT255-project/Modeller/mnist_1_Qkeras_6bw.h5')

/home/slopin/DAT255-project/.venv_qkeras2/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
/home/slopin/DAT255-project/.venv_qkeras2/lib/python3.12/site-packages/tf_keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


NameError: name 'sys' is not defined